In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    sum,
    countDistinct,
    desc,
    round
)

In [3]:
spark=(
    SparkSession.builder
    .appName("Product Analytics")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 17:38:03 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.174 instead (on interface en0)
26/06/13 17:38:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 17:38:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
sales_df=spark.read.parquet(
    "output/gold/sales_fact"
)
sales_df.show(5)
sales_df.printSchema()

+--------------------+--------------------+--------------------+--------------------+------------+-------------+-----+-------------+-------------------+-------------+--------------+------------+----------------+----------------+
|            order_id|         customer_id|          product_id|           seller_id|payment_type|payment_value|price|freight_value| purchase_timestamp|purchase_year|purchase_month|purchase_day|purchase_quarter|purchase_weekday|
+--------------------+--------------------+--------------------+--------------------+------------+-------------+-----+-------------+-------------------+-------------+--------------+------------+----------------+----------------+
|0a2407e0197d11314...|d2038fc66997336ea...|70b487a87fee25bc5...|da8622b14eb17ae28...| credit_card|       201.65|179.9|        21.75|2017-10-19 18:06:14|         2017|            10|          19|               4|        Thursday|
|45934f084d862eb87...|82c8e9be693420696...|29c02f0c592b76b95...|9d4db00d65d776064...

In [ ]:
# identifies your top-performing products and categories by calculating total sales value and order volume for each unique product.

product_df = (
    sales_df
    .groupBy(
        "product_id"
    )
    .agg(
        round(
            sum("price"),
            2
        ).alias("total_revenue"),

        countDistinct(
            "order_id"
        ).alias("orders")
    )
)
product_df.show(5)
product_df.printSchema()

+--------------------+-------------+------+
|          product_id|total_revenue|orders|
+--------------------+-------------+------+
|0b0172eb0fd18479d...|       404.28|    13|
|460a66fcc404a3d73...|        151.5|     6|
|30360c8b0b2ac6918...|        569.7|     3|
|d06466afa11453b64...|       2710.0|    13|
|ed464125465d8ab04...|        900.8|     5|
+--------------------+-------------+------+
only showing top 5 rows
root
 |-- product_id: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- orders: long (nullable = false)



In [11]:
# displaying top product based on revenue

top_products_df = (
    product_df
    .orderBy(
        desc("total_revenue")
    )
)

print("\nTop Products")

top_products_df.show(
    20,
    truncate=False
)
top_products_df.printSchema()



Top Products


+--------------------------------+-------------+------+
|product_id                      |total_revenue|orders|
+--------------------------------+-------------+------+
|bb50f2e236e5eea0100680137654686c|68485.0      |187   |
|d6160fb7873f184099d9bc95e30376af|56948.83     |35    |
|6cdd53843498f92890544667809f1595|55779.9      |151   |
|d1c427060a0f73f6b889a5c7c61f2ac4|49141.4      |323   |
|99a4788cb24856965c36a24e339b6058|46308.96     |467   |
|25c38557cf793876c5abdd5931f922db|44829.32     |38    |
|3dd2a17168ec895c781a9191c1e95ad7|41682.2      |255   |
|aca2eb7d00ea1a7b8ebd4e68314663af|38248.2      |431   |
|53b36df67ebb7c41585e8d54d6772e08|38158.21     |306   |
|5f504b3a1c75b73d6151be81eb05bdc9|37733.9      |63    |
|f819f0c84a64f02d3a5606ca95edd272|35969.48     |45    |
|e0d64dcfaa3b6db5c54ca298ae101d05|33001.33     |194   |
|588531f8ec37e7d5ff5b7b22ea0488f8|32602.99     |19    |
|d285360f29ac7fd97640bf0baef03de0|32227.68     |122   |
|f1c7f353075ce59d8a6f3cf58f419c9c|31724.96     |

In [12]:
# saving the top product which generate the most revenue 

top_products_df.write.mode(
    "overwrite"
).parquet("output/analytics/top_products")

print("\nProduct Analytics Completed")


Product Analytics Completed
